In [ ]:
from langchain_openai import ChatOpenAI
from langchain_google_vertexai import ChatVertexAI

from wsd.load_data import load_data
from wsd.models import BinaryWSD, ClusterByMeaningModel, DummyComparator
from linpub.metrics import accuracy

In [ ]:
X, y = load_data(lang='ja')

k = 200
X_test, y_test = X[:k], y[:k]

In [ ]:
gpt = ChatOpenAI(temperature=0, model="gpt-4o-mini")\
    .with_structured_output(BinaryWSD)
model1 = ClusterByMeaningModel(comparator=gpt)

gemini = ChatVertexAI(temperature=0, model="gemini-1.5-flash")\
    .with_structured_output(BinaryWSD)
model2 = ClusterByMeaningModel(comparator=gemini)

dummy = DummyComparator(probability=1)
model3 = ClusterByMeaningModel(comparator=dummy)

dummy = DummyComparator(probability=0)
model4 = ClusterByMeaningModel(comparator=dummy)

y_pred1 = model1.predict(X_test, verbose=True)
y_pred2 = model2.predict(X_test, verbose=True)
y_pred3 = model3.predict(X_test, verbose=True)
y_pred4 = model4.predict(X_test, verbose=True)

print(f"gpt-4o-mini:  {accuracy(y_pred1, y_test)}")
print(f"gemini1.5-flash: {accuracy(y_pred2, y_test)}")
print(f"dummy (always 1): {accuracy(y_pred3, y_test)}")
print(f"dummy (always 0): {accuracy(y_pred4, y_test)}")

In [ ]:
import pandas as pd

records = []
for yt, yp, x in zip(y_test, y_pred1, X_test):
    record = {'lemma': x.lemma, 'pos': x.pos, 'y': yt, 'y_pred': yp, 'text': x.text, 'context': x.context}
    records.append(record)
pd.DataFrame(records).sort_values(['lemma', 'pos']).to_csv('plop.csv', index=False)